In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.contingency_tables import Table2x2

# Tabela: linhas = horas extras; colunas = Attrition
tabela = pd.DataFrame(
    [[289, 127],
     [944, 110]],
    index=["Horas extras: Sim", "Horas extras: Não"],
    columns=["Attrition: No", "Attrition: Yes"]
)

display(tabela)

# Teste Qui-quadrado
chi2, p_value, dof, expected = chi2_contingency(
    tabela,
    correction=False
)

# Odds Ratio e intervalo de confiança
tabela_2x2 = Table2x2(tabela.values)
odds_ratio = tabela_2x2.oddsratio
ic = tabela_2x2.oddsratio_confint()

print("Qui-quadrado:", chi2)
print("P-value:", p_value)
print("Odds Ratio:", odds_ratio)
print("IC 95%:", ic)

,Attrition: No,Attrition: Yes
Horas extras: Sim,289,127
Horas extras: Não,944,110


Qui-quadrado: 89.04387862461931
P-value: 3.861517910432905e-21
Odds Ratio: 0.26516415320966236
IC 95%: (np.float64(0.19895245220878055), np.float64(0.3534112164328189))


In [ ]:
print("Odds Ratio corrigida:", 1 / odds_ratio)
print("Intervalo de confiança 95%:", (1 / ic[1], 1 / ic[0]))

Odds Ratio corrigida: 3.7712488203837684
Intervalo de confiança 95%: (np.float64(2.8295649755929384), np.float64(5.026326586568537))


In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

arquivo = next(iter(uploaded))

df = pd.read_csv(io.BytesIO(uploaded[arquivo]))

print("Dimensões da base:", df.shape)

display(df.head())

print("\nTipos de dados:")
display(df.dtypes)

KeyboardInterrupt: 

In [ ]:
# 1. Verificar dimensões
print("Dimensões:", df.shape)

# 2. Verificar valores ausentes
print("\nValores ausentes:")
print(df.isnull().sum().sum())

# 3. Verificar duplicidades
print("\nLinhas duplicadas:")
print(df.duplicated().sum())

# 4. Verificar distribuição da variável-alvo
print("\nDistribuição de Attrition:")
print(df["Attrition"].value_counts())

print("\nPercentual de Attrition:")
print(
    (df["Attrition"].value_counts(normalize=True) * 100)
    .round(2)
)

# 5. Identificar colunas constantes
print("\nColunas constantes:")
print(df.columns[df.nunique() <= 1].tolist())

In [ ]:

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# 1. Definir variáveis que não entrarão no modelo
colunas_excluir = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

# 2. Separar features e target
X = df.drop(columns=colunas_excluir + ["Attrition"])

y = df["Attrition"].map({
    "No": 0,
    "Yes": 1
})

# 3. Identificar os tipos de variáveis
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Variáveis numéricas:", len(numeric_features))
print("Variáveis categóricas:", len(categorical_features))
print("Total de features:", X.shape[1])

# 4. Preparar transformações
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

# 5. Separar treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTreinamento:", X_train.shape)
print("Teste:", X_test.shape)

# 6. Construir pipeline
modelo_logistico = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]
)

# 7. Treinar modelo
modelo_logistico.fit(X_train, y_train)

# 8. Fazer previsões
y_pred = modelo_logistico.predict(X_test)
y_prob = modelo_logistico.predict_proba(X_test)[:, 1]

# 9. Avaliar desempenho
print("\n--- RESULTADOS DO MODELO ---")

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_test, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\n--- MATRIZ DE CONFUSÃO ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Attrition", "Attrition"],
    zero_division=0
))

In [ ]:

# Modelo 2: Logistic Regression com classes balanceadas

modelo_balanceado = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

# Treinamento
modelo_balanceado.fit(X_train, y_train)

# Previsões
y_pred_bal = modelo_balanceado.predict(X_test)
y_prob_bal = modelo_balanceado.predict_proba(X_test)[:, 1]

# Avaliação
print("--- MODELO 2: BALANCED ---")

print("Accuracy:", round(
    accuracy_score(y_test, y_pred_bal), 4
))

print("Precision:", round(
    precision_score(y_test, y_pred_bal), 4
))

print("Recall:", round(
    recall_score(y_test, y_pred_bal), 4
))

print("F1-score:", round(
    f1_score(y_test, y_pred_bal), 4
))

print("ROC-AUC:", round(
    roc_auc_score(y_test, y_prob_bal), 4
))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred_bal))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_bal,
    target_names=["No Attrition", "Attrition"],
    zero_division=0
))


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve
)
import pandas as pd
import numpy as np

# 1. Separar treinamento interno e validação
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

# 2. Treinar modelo balanceado apenas no subtreino
modelo_validacao = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

modelo_validacao.fit(X_subtrain, y_subtrain)

# 3. Obter probabilidades na validação
prob_val = modelo_validacao.predict_proba(X_val)[:, 1]

# 4. Comparar diferentes thresholds
thresholds = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

resultados = []

for threshold in thresholds:
    pred_val = (prob_val >= threshold).astype(int)

    resultados.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, pred_val, zero_division=0
        ),
        "Recall": recall_score(
            y_val, pred_val, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, pred_val, zero_division=0
        )
    })

tabela_thresholds = pd.DataFrame(resultados)

display(
    tabela_thresholds.round(4)
)

# 5. Escolher o threshold com maior F1 na validação
melhor_indice = tabela_thresholds["F1-score"].idxmax()

melhor_threshold = tabela_thresholds.loc[
    melhor_indice, "Threshold"
]

print("Threshold selecionado na validação:", melhor_threshold)


In [ ]:

# 1. Criar o modelo final balanceado
modelo_final = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

# 2. Treinar com todo o conjunto de treinamento
modelo_final.fit(X_train, y_train)

# 3. Calcular probabilidades no conjunto de teste
prob_final = modelo_final.predict_proba(X_test)[:, 1]

# 4. Aplicar o threshold selecionado na validação
threshold_final = 0.60

y_pred_final = (
    prob_final >= threshold_final
).astype(int)

# 5. Avaliação final
print("--- MODELO FINAL ---")
print("Threshold:", threshold_final)

print("Accuracy:",
      round(accuracy_score(y_test, y_pred_final), 4))

print("Precision:",
      round(precision_score(y_test, y_pred_final, zero_division=0), 4))

print("Recall:",
      round(recall_score(y_test, y_pred_final, zero_division=0), 4))

print("F1-score:",
      round(f1_score(y_test, y_pred_final, zero_division=0), 4))

print("ROC-AUC:",
      round(roc_auc_score(y_test, prob_final), 4))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred_final))

print("\nRelatório de classificação:")
print(classification_report(
    y_test,
    y_pred_final,
    target_names=["No Attrition", "Attrition"],
    zero_division=0
))


In [ ]:

import pandas as pd
import numpy as np

# 1. Recuperar os nomes das variáveis após o processamento
nomes_features = (
    modelo_final
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# 2. Recuperar os coeficientes da regressão logística
coeficientes = (
    modelo_final
    .named_steps["classifier"]
    .coef_[0]
)

# 3. Criar tabela de interpretabilidade
tabela_odds = pd.DataFrame({
    "Variavel": nomes_features,
    "Coeficiente": coeficientes,
    "Odds_Ratio": np.exp(coeficientes)
})

# 4. Ordenar pelas maiores associações positivas
tabela_odds = tabela_odds.sort_values(
    by="Odds_Ratio",
    ascending=False
)

# 5. Exibir resultados
display(
    tabela_odds.round(4)
)


In [ ]:

# Recriar o pré-processador com categoria de referência

preprocessor_ref = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

# Criar novo modelo balanceado
modelo_interpretavel = Pipeline(
    steps=[
        ("preprocessor", preprocessor_ref),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

# Treinar usando o conjunto de treinamento
modelo_interpretavel.fit(X_train, y_train)

# Extrair nomes e coeficientes
nomes = (
    modelo_interpretavel
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

coef = (
    modelo_interpretavel
    .named_steps["classifier"]
    .coef_[0]
)

# Construir tabela de Odds Ratios
tabela_odds_ref = pd.DataFrame({
    "Variavel": nomes,
    "Coeficiente": coef,
    "Odds_Ratio": np.exp(coef)
})

# Ordenar e exibir
tabela_odds_ref = tabela_odds_ref.sort_values(
    "Odds_Ratio",
    ascending=False
)

display(tabela_odds_ref.round(4))


In [ ]:

from sklearn.model_selection import StratifiedKFold, cross_validate
import pandas as pd
import numpy as np

# 1. Configurar validação cruzada
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 2. Definir métricas
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# 3. Executar validação cruzada
resultados_cv = cross_validate(
    modelo_interpretavel,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# 4. Organizar resultados
metricas = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": resultados_cv["test_accuracy"],
    "Precision": resultados_cv["test_precision"],
    "Recall": resultados_cv["test_recall"],
    "F1-score": resultados_cv["test_f1"],
    "ROC-AUC": resultados_cv["test_roc_auc"]
})

display(metricas.round(4))

# 5. Calcular média e desvio padrão
resumo_cv = pd.DataFrame({
    "Média": metricas.drop(columns="Fold").mean(),
    "Desvio_Padrao": metricas.drop(columns="Fold").std()
})

print("RESUMO DA VALIDAÇÃO CRUZADA")
display(resumo_cv.round(4))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
import pandas as pd

modelo_sem_balanceamento = Pipeline([
    ("preprocessor", preprocessor_ref),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

resultados_sem_balanceamento = cross_validate(
    modelo_sem_balanceamento,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

comparacao = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Com balanceamento": [
        0.7483, 0.3640, 0.7316, 0.4848, 0.8268
    ],
    "Sem balanceamento": [
        resultados_sem_balanceamento["test_accuracy"].mean(),
        resultados_sem_balanceamento["test_precision"].mean(),
        resultados_sem_balanceamento["test_recall"].mean(),
        resultados_sem_balanceamento["test_f1"].mean(),
        resultados_sem_balanceamento["test_roc_auc"].mean()
    ]
})

display(comparacao.round(4))

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Modelo sem balanceamento
modelo_sem_balanceamento.fit(X_train, y_train)

pred_sem = modelo_sem_balanceamento.predict(X_test)

# Modelo com balanceamento
modelo_balanceado = Pipeline([
    ("preprocessor", preprocessor_ref),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42,
        class_weight="balanced"
    ))
])

modelo_balanceado.fit(X_train, y_train)

pred_com = modelo_balanceado.predict(X_test)

print("MODELO SEM BALANCEAMENTO")
print(confusion_matrix(y_test, pred_sem))
print(classification_report(y_test, pred_sem))

print("\nMODELO COM BALANCEAMENTO")
print(confusion_matrix(y_test, pred_com))
print(classification_report(y_test, pred_com))

In [ ]:

!pip -q install statsmodels

import numpy as np
import pandas as pd
import statsmodels.api as sm

# 1. Preparar os dados usando o pré-processador já definido
X_train_processado = preprocessor_ref.fit_transform(X_train)

# Converter para matriz densa, se necessário
if hasattr(X_train_processado, "toarray"):
    X_train_processado = X_train_processado.toarray()

# 2. Recuperar os nomes das variáveis
nomes_variaveis = preprocessor_ref.get_feature_names_out()

X_inferencia = pd.DataFrame(
    X_train_processado,
    columns=nomes_variaveis,
    index=X_train.index
)

# 3. Adicionar intercepto
X_inferencia = sm.add_constant(X_inferencia)

# 4. Ajustar regressão logística
modelo_inferencial = sm.Logit(
    y_train.astype(int),
    X_inferencia
)

resultado_inferencial = modelo_inferencial.fit(
    maxiter=200,
    disp=False
)

# 5. Calcular Odds Ratios e intervalos de confiança
coeficientes = resultado_inferencial.params
intervalos = resultado_inferencial.conf_int()
p_valores = resultado_inferencial.pvalues

tabela_or = pd.DataFrame({
    "Coeficiente": coeficientes,
    "Odds_Ratio": np.exp(coeficientes),
    "IC_95%_Inferior": np.exp(intervalos[0]),
    "IC_95%_Superior": np.exp(intervalos[1]),
    "P_value": p_valores
})

# 6. Organizar e exibir
tabela_or = tabela_or.drop(index="const", errors="ignore")
tabela_or = tabela_or.sort_values("P_value")

display(tabela_or.round(4))

print("Convergiu:", resultado_inferencial.mle_retvals["converged"])
print("Observações:", int(resultado_inferencial.nobs))
print("Pseudo R²:", round(resultado_inferencial.prsquared, 4))


In [ ]:

import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Retirar Department apenas da análise inferencial
X_inferencia_base = X_train.drop(columns=["Department"])

# 2. Identificar os tipos de variáveis
numeric_inf = X_inferencia_base.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_inf = X_inferencia_base.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# 3. Criar novo pré-processador
preprocessor_inf = ColumnTransformer([
    ("num", StandardScaler(), numeric_inf),
    ("cat", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ), categorical_inf)
])

# 4. Transformar os dados
X_processado_inf = preprocessor_inf.fit_transform(
    X_inferencia_base
)

if hasattr(X_processado_inf, "toarray"):
    X_processado_inf = X_processado_inf.toarray()

nomes_inf = preprocessor_inf.get_feature_names_out()

X_inf = pd.DataFrame(
    X_processado_inf,
    columns=nomes_inf,
    index=X_train.index
)

# 5. Adicionar intercepto
X_inf = sm.add_constant(X_inf)

# 6. Ajustar modelo
modelo_logit_inf = sm.Logit(
    y_train.astype(int),
    X_inf
)

resultado_inf = modelo_logit_inf.fit(
    maxiter=500,
    disp=False
)

# 7. Verificar convergência
print("Convergiu:", resultado_inf.mle_retvals["converged"])
print("Observações:", int(resultado_inf.nobs))
print("Pseudo R²:", round(resultado_inf.prsquared, 4))

# 8. Calcular Odds Ratios e IC 95%
coef = resultado_inf.params
ic = resultado_inf.conf_int()

tabela_inferencial = pd.DataFrame({
    "Coeficiente": coef,
    "Odds_Ratio": np.exp(coef),
    "IC_95%_Inferior": np.exp(ic[0]),
    "IC_95%_Superior": np.exp(ic[1]),
    "P_value": resultado_inf.pvalues
})

tabela_inferencial = tabela_inferencial.drop(
    index="const",
    errors="ignore"
)

tabela_inferencial = tabela_inferencial.sort_values(
    "P_value"
)

display(tabela_inferencial.round(4))


In [ ]:

from statsmodels.stats.multitest import multipletests

# Copiar tabela de resultados
tabela_fdr = tabela_inferencial.copy()

# Aplicar correção Benjamini-Hochberg
rejeitar, p_ajustados, _, _ = multipletests(
    tabela_fdr["P_value"],
    alpha=0.05,
    method="fdr_bh"
)

tabela_fdr["P_value_ajustado"] = p_ajustados
tabela_fdr["Significativo_FDR"] = rejeitar

# Ordenar pelo p-valor ajustado
tabela_fdr = tabela_fdr.sort_values(
    "P_value_ajustado"
)

display(tabela_fdr.round(4))

print(
    "Variáveis significativas após FDR:",
    int(tabela_fdr["Significativo_FDR"].sum())
)


In [ ]:

from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import numpy as np

# Usar a matriz já processada, sem a constante
X_vif = X_inf.drop(columns=["const"]).copy()

# Calcular VIF
resultados_vif = pd.DataFrame({
    "Variavel": X_vif.columns,
    "VIF": [
        variance_inflation_factor(
            X_vif.values, i
        )
        for i in range(X_vif.shape[1])
    ]
})

resultados_vif = resultados_vif.sort_values(
    "VIF", ascending=False
)

display(resultados_vif.round(2))

print("\nVariáveis com VIF acima de 5:")
display(
    resultados_vif[
        resultados_vif["VIF"] > 5
    ].round(2)
)


In [ ]:

from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

# Usar a matriz com constante
X_vif_corrigido = X_inf.copy()

# Calcular VIF incluindo a constante na regressão auxiliar
resultados_vif_corrigidos = pd.DataFrame({
    "Variavel": X_vif_corrigido.columns[1:],
    "VIF": [
        variance_inflation_factor(
            X_vif_corrigido.values, i
        )
        for i in range(1, X_vif_corrigido.shape[1])
    ]
})

resultados_vif_corrigidos = resultados_vif_corrigidos.sort_values(
    "VIF", ascending=False
)

display(resultados_vif_corrigidos.round(2))

print("\nVariáveis com VIF acima de 5:")
display(
    resultados_vif_corrigidos[
        resultados_vif_corrigidos["VIF"] > 5
    ].round(2)
)


In [ ]:
print("Dimensões:", X_inf.shape)
print("Constante presente:", "const" in X_inf.columns)
print("Primeiras colunas:", X_inf.columns[:5].tolist())

print("\nMédia da constante:")
print(X_inf["const"].mean())

print("\nPrimeiros valores da constante:")
print(X_inf["const"].head())

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import numpy as np

X_vif_final = X_inf.copy()

vif_final = []

for i, coluna in enumerate(X_vif_final.columns):
    if coluna == "const":
        continue

    valor_vif = variance_inflation_factor(
        X_vif_final.values,
        i
    )

    vif_final.append({
        "Variavel": coluna,
        "VIF": valor_vif
    })

tabela_vif_final = pd.DataFrame(vif_final)

tabela_vif_final = tabela_vif_final.sort_values(
    "VIF", ascending=False
)

display(tabela_vif_final.round(3))

print("Maior VIF:", round(tabela_vif_final["VIF"].max(), 3))
print(
    "Quantidade acima de 5:",
    (tabela_vif_final["VIF"] > 5).sum()
)


In [ ]:
print("df existe:", "df" in globals())
print("X_train existe:", "X_train" in globals())
print("y_train existe:", "y_train" in globals())

In [ ]:
from google.colab import files

arquivos = files.upload()

In [ ]:
import pandas as pd

nome_arquivo = "WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(nome_arquivo)

print("Dimensões:", df.shape)
display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split

colunas_excluir = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

X = df.drop(columns=colunas_excluir + ["Attrition"])

y = df["Attrition"].map({
    "No": 0,
    "Yes": 1
})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treinamento:", X_train.shape)
print("Teste:", X_test.shape)

In [ ]:
print("df:", df.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
variaveis_relevantes = [
    "MonthlyIncome",
    "JobLevel",
    "TotalWorkingYears",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsWithCurrManager"
]

correlacoes = X_train[variaveis_relevantes].corr()

display(correlacoes.round(3))

In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pandas as pd
import numpy as np

# Remover MonthlyIncome apenas da análise inferencial
X_inferencia_base = X_train.drop(
    columns=["Department", "MonthlyIncome"]
)

# Identificar tipos de variáveis
numeric_inf = X_inferencia_base.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_inf = X_inferencia_base.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Novo pré-processamento
preprocessor_inf = ColumnTransformer([
    ("num", StandardScaler(), numeric_inf),
    ("cat", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ), categorical_inf)
])

# Transformar os dados
X_processado_inf = preprocessor_inf.fit_transform(
    X_inferencia_base
)

if hasattr(X_processado_inf, "toarray"):
    X_processado_inf = X_processado_inf.toarray()

nomes_inf = preprocessor_inf.get_feature_names_out()

X_inf = pd.DataFrame(
    X_processado_inf,
    columns=nomes_inf,
    index=X_train.index
)

# Adicionar constante
import statsmodels.api as sm

X_inf = sm.add_constant(X_inf)

print("Dimensões:", X_inf.shape)
print("Variáveis explicativas:", X_inf.shape[1] - 1)


In [ ]:

from statsmodels.stats.outliers_influence import variance_inflation_factor

resultados_vif = []

for i, coluna in enumerate(X_inf.columns):
    if coluna == "const":
        continue

    valor = variance_inflation_factor(
        X_inf.values, i
    )

    resultados_vif.append({
        "Variavel": coluna,
        "VIF": valor
    })

tabela_vif = pd.DataFrame(resultados_vif)

tabela_vif = tabela_vif.sort_values(
    "VIF", ascending=False
)

display(tabela_vif.round(3))

print(
    "Quantidade acima de 5:",
    (tabela_vif["VIF"] > 5).sum()
)

In [ ]:

# Reconstruir o pré-processador com a referência padrão
preprocessor_inf = ColumnTransformer([
    ("num", StandardScaler(), numeric_inf),
    ("cat", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ), categorical_inf)
])

# Transformar os dados
X_processado_inf = preprocessor_inf.fit_transform(
    X_inferencia_base
)

if hasattr(X_processado_inf, "toarray"):
    X_processado_inf = X_processado_inf.toarray()

nomes_inf = preprocessor_inf.get_feature_names_out()

X_inf = pd.DataFrame(
    X_processado_inf,
    columns=nomes_inf,
    index=X_train.index
)

# Adicionar constante
X_inf = sm.add_constant(X_inf)

print("Dimensões:", X_inf.shape)
print("Pré-processamento reconstruído com sucesso.")


In [ ]:

from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import statsmodels.api as sm

# Selecionar somente variáveis numéricas
colunas_numericas = [
    coluna for coluna in X_inf.columns
    if coluna.startswith("num__")
]

X_vif_numerico = X_inf[colunas_numericas].copy()

# Adicionar constante
X_vif_numerico = sm.add_constant(X_vif_numerico)

# Calcular VIF
vif_numerico = pd.DataFrame({
    "Variavel": colunas_numericas,
    "VIF": [
        variance_inflation_factor(
            X_vif_numerico.values, i + 1
        )
        for i in range(len(colunas_numericas))
    ]
})

vif_numerico = vif_numerico.sort_values(
    "VIF", ascending=False
)

display(vif_numerico.round(3))


In [ ]:

import statsmodels.api as sm
import numpy as np
import pandas as pd

# Ajustar regressão logística inferencial
modelo_final_inf = sm.Logit(
    y_train.astype(int),
    X_inf
)

resultado_final_inf = modelo_final_inf.fit(
    maxiter=500,
    disp=False
)

# Verificar convergência
print("Convergiu:", resultado_final_inf.mle_retvals["converged"])
print("Observações:", int(resultado_final_inf.nobs))
print("Pseudo R²:", round(resultado_final_inf.prsquared, 4))

# Calcular Odds Ratios e intervalos de confiança
coef = resultado_final_inf.params
ic = resultado_final_inf.conf_int()

tabela_final_inf = pd.DataFrame({
    "Coeficiente": coef,
    "Odds_Ratio": np.exp(coef),
    "IC_95%_Inferior": np.exp(ic[0]),
    "IC_95%_Superior": np.exp(ic[1]),
    "P_value": resultado_final_inf.pvalues
})

tabela_final_inf = tabela_final_inf.drop(
    index="const",
    errors="ignore"
)

# Correção de múltiplos testes
from statsmodels.stats.multitest import multipletests

rejeitar, p_ajustados, _, _ = multipletests(
    tabela_final_inf["P_value"],
    alpha=0.05,
    method="fdr_bh"
)

tabela_final_inf["P_value_ajustado"] = p_ajustados
tabela_final_inf["Significativo_FDR"] = rejeitar

tabela_final_inf = tabela_final_inf.sort_values(
    "P_value_ajustado"
)

display(tabela_final_inf.round(4))

print(
    "\nVariáveis significativas após FDR:",
    int(tabela_final_inf["Significativo_FDR"].sum())
)


In [ ]:

# Diagnóstico de influência
influencia = resultado_final_inf.get_influence()

diagnostico = influencia.summary_frame()

# Distância de Cook
cooks_d = diagnostico["cooks_d"]

# Referência exploratória
limite_cook = 4 / len(X_inf)

print("Limite de referência:", round(limite_cook, 5))
print("Maior distância de Cook:", round(cooks_d.max(), 5))
print("Observações acima do limite:", int((cooks_d > limite_cook).sum()))

# Dez observações mais influentes
display(
    diagnostico[["cooks_d", "standard_resid"]]
    .sort_values("cooks_d", ascending=False)
    .head(10)
    .round(4)
)


In [ ]:

# Recuperar os índices das 10 observações mais influentes
top_indices = cooks_d.sort_values(
    ascending=False
).head(10).index

# Características relevantes da base original
colunas_analise = [
    "Attrition",
    "Age",
    "Department",
    "JobRole",
    "OverTime",
    "BusinessTravel",
    "MonthlyIncome",
    "JobLevel",
    "TotalWorkingYears",
    "YearsAtCompany",
    "YearsSinceLastPromotion",
    "JobSatisfaction",
    "EnvironmentSatisfaction"
]

# Combinar dados originais e diagnóstico
casos_influentes = df.loc[
    top_indices, colunas_analise
].copy()

casos_influentes["Cooks_D"] = cooks_d.loc[top_indices]
casos_influentes["Resíduo_Padronizado"] = (
    diagnostico.loc[top_indices, "standard_resid"]
)

casos_influentes = casos_influentes.sort_values(
    "Cooks_D", ascending=False
)

display(casos_influentes.round(4))


In [ ]:

# Identificar os 10 casos mais influentes
top10_indices = cooks_d.nlargest(10).index

# Criar amostra de diagnóstico, sem alterar a base original
X_sensibilidade = X_inf.drop(index=top10_indices)
y_sensibilidade = y_train.drop(index=top10_indices)

# Ajustar modelo alternativo
modelo_sensibilidade = sm.Logit(
    y_sensibilidade.astype(int),
    X_sensibilidade
)

resultado_sensibilidade = modelo_sensibilidade.fit(
    maxiter=500,
    disp=False
)

# Comparar coeficientes
comparacao_coef = pd.DataFrame({
    "Coef_Completo": resultado_final_inf.params,
    "Coef_Sem_Top10": resultado_sensibilidade.params
})

comparacao_coef["Diferenca_Absoluta"] = (
    comparacao_coef["Coef_Sem_Top10"]
    - comparacao_coef["Coef_Completo"]
).abs()

comparacao_coef["Variacao_Percentual"] = (
    comparacao_coef["Diferenca_Absoluta"]
    / comparacao_coef["Coef_Completo"].abs().replace(0, np.nan)
) * 100

display(
    comparacao_coef.sort_values(
        "Variacao_Percentual",
        ascending=False
    ).round(3)
)

print(
    "Convergiu:",
    resultado_sensibilidade.mle_retvals["converged"]
)

print(
    "Observações:",
    int(resultado_sensibilidade.nobs)
)


In [ ]:
top10_indices = cooks_d.nlargest(10).index

y_sensibilidade = y_train.drop(index=top10_indices)

X_sensibilidade_base = X_train.drop(
    index=top10_indices
)

diagnostico_cargos = pd.crosstab(
    X_sensibilidade_base["JobRole"],
    y_sensibilidade.map({0: "No", 1: "Yes"})
)

diagnostico_cargos["Total"] = diagnostico_cargos.sum(axis=1)

diagnostico_cargos

In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import statsmodels.api as sm

# Carregar arquivo
df = pd.read_csv(
    "/content/WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

# Separar variáveis explicativas e alvo
colunas_excluir = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

X = df.drop(columns=colunas_excluir + ["Attrition"])

y = df["Attrition"].map({
    "No": 0,
    "Yes": 1
})

# Divisão estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Base:", df.shape)
print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


In [ ]:

from google.colab import files

arquivos = files.upload()


In [ ]:

import pandas as pd
import io

nome_arquivo = next(
    nome for nome in arquivos
    if nome.endswith(".csv")
)

df = pd.read_csv(
    io.BytesIO(arquivos[nome_arquivo])
)

print("Dimensões:", df.shape)
print(df.head())


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import statsmodels.api as sm
import numpy as np
import pandas as pd

colunas_excluir = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

X = df.drop(columns=colunas_excluir + ["Attrition"])
y = df["Attrition"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Base completa:", df.shape)
print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print("Distribuição do alvo:")
print(y.value_counts())


In [ ]:

# Excluir variáveis retiradas na especificação final
X_inferencia_base = X_train.drop(
    columns=["Department", "MonthlyIncome"]
)

# Identificar tipos de variáveis
numeric_inf = X_inferencia_base.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_inf = X_inferencia_base.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Pré-processamento
preprocessor_inf = ColumnTransformer([
    ("num", StandardScaler(), numeric_inf),
    ("cat", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ), categorical_inf)
])

X_processado_inf = preprocessor_inf.fit_transform(
    X_inferencia_base
)

if hasattr(X_processado_inf, "toarray"):
    X_processado_inf = X_processado_inf.toarray()

nomes_inf = preprocessor_inf.get_feature_names_out()

X_inf = pd.DataFrame(
    X_processado_inf,
    columns=nomes_inf,
    index=X_train.index
)

# Adicionar constante
X_inf = sm.add_constant(X_inf)

# Ajustar modelo
modelo_final_inf = sm.Logit(
    y_train.astype(int),
    X_inf
)

resultado_final_inf = modelo_final_inf.fit(
    maxiter=500,
    disp=False
)

print("Convergiu:", resultado_final_inf.mle_retvals["converged"])
print("Observações:", int(resultado_final_inf.nobs))
print("Pseudo R²:", round(resultado_final_inf.prsquared, 4))


In [ ]:

from statsmodels.stats.multitest import multipletests

# Extrair resultados
coeficientes = resultado_final_inf.params
intervalos = resultado_final_inf.conf_int()
p_valores = resultado_final_inf.pvalues

# Construir tabela
tabela_regressao = pd.DataFrame({
    "Variavel": coeficientes.index,
    "Coeficiente": coeficientes.values,
    "OR": np.exp(coeficientes.values),
    "IC95_Inferior": np.exp(intervalos.iloc[:, 0].values),
    "IC95_Superior": np.exp(intervalos.iloc[:, 1].values),
    "Valor_p": p_valores.values
})

# Aplicar correção FDR, excluindo a constante
mascara = tabela_regressao["Variavel"] != "const"

tabela_regressao["Valor_p_ajustado"] = np.nan

tabela_regressao.loc[mascara, "Valor_p_ajustado"] = (
    multipletests(
        tabela_regressao.loc[mascara, "Valor_p"],
        method="fdr_bh"
    )[1]
)

# Identificar significância estatística
tabela_regressao["Significativo_FDR"] = (
    tabela_regressao["Valor_p_ajustado"] < 0.05
)

# Ordenar pelos valores ajustados
tabela_regressao = tabela_regressao.sort_values(
    "Valor_p_ajustado",
    na_position="last"
)

# Arredondar resultados
colunas_numericas = [
    "Coeficiente",
    "OR",
    "IC95_Inferior",
    "IC95_Superior",
    "Valor_p",
    "Valor_p_ajustado"
]

tabela_regressao[colunas_numericas] = (
    tabela_regressao[colunas_numericas].round(4)
)

print(
    "Variáveis significativas após FDR:",
    tabela_regressao["Significativo_FDR"].sum()
)

tabela_regressao


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Selecionar apenas variáveis significativas
dados_grafico = tabela_regressao[
    tabela_regressao["Significativo_FDR"] == True
].copy()

# Ordenar pelo OR
dados_grafico = dados_grafico.sort_values("OR")

# Criar gráfico
plt.figure(figsize=(10, 8))

plt.errorbar(
    dados_grafico["OR"],
    dados_grafico["Variavel"],
    xerr=[
        dados_grafico["OR"] - dados_grafico["IC95_Inferior"],
        dados_grafico["IC95_Superior"] - dados_grafico["OR"]
    ],
    fmt="o",
    capsize=4
)

plt.axvline(
    x=1,
    linestyle="--",
    linewidth=1
)

plt.xscale("log")
plt.xlabel("Odds Ratio (escala logarítmica)")
plt.ylabel("Variáveis")
plt.title("Fatores associados à Attrition — Regressão Logística")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

from google.colab import files
arquivos = files.upload()


Saving WA_Fn-UseC_-HR-Employee-Attrition.csv to WA_Fn-UseC_-HR-Employee-Attrition.csv


In [ ]:

import pandas as pd
import numpy as np
import io

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import statsmodels.api as sm

nome_arquivo = next(
    nome for nome in arquivos if nome.endswith(".csv")
)

df = pd.read_csv(io.BytesIO(arquivos[nome_arquivo]))

colunas_excluir = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

X = df.drop(columns=colunas_excluir + ["Attrition"])
y = df["Attrition"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Base:", df.shape)
print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


Base: (1470, 35)
Treino: (1176, 30)
Teste: (294, 30)


In [ ]:

X_inferencia_base = X_train.drop(
    columns=["Department", "MonthlyIncome"]
)

numeric_inf = X_inferencia_base.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_inf = X_inferencia_base.select_dtypes(
    include=["object", "category"]
).columns.tolist()

preprocessor_inf = ColumnTransformer([
    ("num", StandardScaler(), numeric_inf),
    ("cat", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ), categorical_inf)
])

X_processado_inf = preprocessor_inf.fit_transform(
    X_inferencia_base
)

if hasattr(X_processado_inf, "toarray"):
    X_processado_inf = X_processado_inf.toarray()

nomes_inf = preprocessor_inf.get_feature_names_out()

X_inf = pd.DataFrame(
    X_processado_inf,
    columns=nomes_inf,
    index=X_train.index
)

X_inf = sm.add_constant(X_inf)

modelo_final_inf = sm.Logit(
    y_train.astype(int),
    X_inf
)

resultado_final_inf = modelo_final_inf.fit(
    maxiter=500,
    disp=False
)

print("Convergiu:", resultado_final_inf.mle_retvals["converged"])
print("Observações:", int(resultado_final_inf.nobs))
print("Pseudo R²:", round(resultado_final_inf.prsquared, 4))


Convergiu: True
Observações: 1176
Pseudo R²: 0.3581


In [ ]:

from statsmodels.stats.multitest import multipletests

coef = resultado_final_inf.params
ic = resultado_final_inf.conf_int()
p = resultado_final_inf.pvalues

tabela_regressao = pd.DataFrame({
    "Variavel": coef.index,
    "Coeficiente": coef.values,
    "OR": np.exp(coef.values),
    "IC95_Inferior": np.exp(ic.iloc[:, 0].values),
    "IC95_Superior": np.exp(ic.iloc[:, 1].values),
    "Valor_p": p.values
})

mascara = tabela_regressao["Variavel"] != "const"

tabela_regressao["Valor_p_ajustado"] = np.nan

tabela_regressao.loc[mascara, "Valor_p_ajustado"] = (
    multipletests(
        tabela_regressao.loc[mascara, "Valor_p"],
        method="fdr_bh"
    )[1]
)

tabela_regressao["Significativo_FDR"] = (
    tabela_regressao["Valor_p_ajustado"] < 0.05
)

tabela_regressao = tabela_regressao.sort_values(
    "Valor_p_ajustado",
    na_position="last"
)

tabela_regressao.to_csv(
    "resultados_regressao_logistica.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Variáveis significativas:",
      tabela_regressao["Significativo_FDR"].sum())

print("Arquivo salvo com sucesso!")

Variáveis significativas: 15
Arquivo salvo com sucesso!


In [ ]:
from google.colab import files
files.download("resultados_regressao_logistica.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print([
    nome for nome in globals()
    if any(p in nome.lower() for p in [
        "modelo", "threshold", "y_pred",
        "y_prob", "metric", "balanced"
    ])
])

['modelo_final_inf']


In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Preparação das variáveis
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

# Modelo balanceado
modelo_balanceado = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

modelo_balanceado.fit(X_train, y_train)

# Probabilidades no conjunto de teste
probabilidades = modelo_balanceado.predict_proba(X_test)[:, 1]

# Threshold definido anteriormente
threshold = 0.60

y_pred = (probabilidades >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, probabilidades))
print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.8129251700680272
Precision: 0.43548387096774194
Recall: 0.574468085106383
F1: 0.4954128440366973
ROC-AUC: 0.8031699543457662
Matriz de confusão:
[[212  35]
 [ 20  27]]


In [ ]:

import pandas as pd
from google.colab import files

# Métricas finais
metricas_finais = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC",
        "Threshold"
    ],
    "Value": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, probabilidades),
        0.60
    ]
})

# Matriz de confusão
matriz = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["Actual_No", "Actual_Yes"],
    columns=["Predicted_No", "Predicted_Yes"]
)

metricas_finais.to_csv(
    "metricas_modelo_preditivo.csv",
    index=False
)

matriz.to_csv("matriz_confusao.csv")

print(metricas_finais)
print("\nMatriz de confusão:")
print(matriz)

files.download("metricas_modelo_preditivo.csv")
files.download("matriz_confusao.csv")


      Metric     Value
0   Accuracy  0.812925
1  Precision  0.435484
2     Recall  0.574468
3   F1-score  0.495413
4    ROC-AUC  0.803170
5  Threshold  0.600000

Matriz de confusão:
            Predicted_No  Predicted_Yes
Actual_No            212             35
Actual_Yes            20             27


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>